In [ ]:
#Scaling + Pipeline
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Function to create dataset
def make_student_success(n=600, seed=1):

    rng = np.random.default_rng(seed)

    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })

    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)

    # Add noise (real-life mistakes)
    y = np.where(rng.random(n) < 0.07, 1-y, y)

    return X, y


# Create dataset
X, y = make_student_success()

#print(X)



#show the raw data 

print(X.head())
print(y[:5])



# train and test Split

X_train, X_test, y_train, y_test = train_test_split(
            X, y,test_size=0.20,random_state=3,stratify=y
        )


# Pipeline 
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model",LogisticRegression(max_iter=1000) )
]
)


#train the model 
model.fit(X_train, y_train)
# test model 
pred = model.predict(X_test)

print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

# Predict New Students
new_student = [[6, 100, 7, 0]]

print("Prediction : " , model.predict(new_student)[0])

## 4️⃣ Scaling — Why It Exists

Look at your features:

| Feature         | Range     |
|-----------------|-----------|
| hours           | 0 – 12    |
| attendance      | 40 – 100  |
| sleep           | 4 – 9     |
| practice_tests  | 0 – 6     |

They are on **different scales**.

Logistic Regression calculates **weights based on feature magnitude**.

If one feature has a **much larger numeric range**, it can **dominate the learning process**.

## What Scaling Does

`StandardScaler` transforms each feature so that:

- **Mean = 0**
- **Standard Deviation = 1**

### Mathematically
z = (x - μ) / σ

Where:

- **x** = original value  
- **μ** = mean of the feature  
- **σ** = standard deviation  

## Why This Matters

### Without Scaling

The model might **over-prioritize `attendance`** simply because it contains **larger numeric values**.

The algorithm may interpret larger numbers as **more important**, even when they are not.

---

### With Scaling

All features are brought to a **comparable scale**.

The model focuses on **real relationships in the data**, not the **size of the numbers**.

---

## Important

**Tree-based models usually do NOT need scaling**

Examples:

- Decision Trees  
- Random Forest  

**Linear and distance-based models often benefit from scaling**

Examples:

- Logistic Regression  
- Linear Regression  
- Support Vector Machines (SVM)


# Training vs Testing, Scaling, and Pipelines

## What is a Pipeline?

This is where many beginners **silently break Machine Learning**.

---

## Common Mistake

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # ❌ WRONG place
```

If you scale **before splitting the data**, you accidentally allow the model to see information from the **test set**.

You just used **test data statistics** to help train the model.

This is called **data leakage**.

It makes your **test results falsely optimistic**.

The model appears better than it actually is.

---

## What a Pipeline Does

```python
clf = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

A Pipeline ensures the correct order of operations.

### Proper Workflow

1. Split the data
2. Fit the scaler **only on training data**
3. Apply the same scaler to test data
4. Train the model

All of this happens **automatically**.

This prevents accidental data leakage.

---

## Mental Model

### Without Pipeline

You manually control every step.

Easy to forget something.  
Easy to make mistakes.

---

### With Pipeline

You **define the process once**.

`sklearn` controls the **execution order**.

This enforces **discipline and correctness**.

---

# Why Scaling + Pipeline Together is Powerful

When you do:

```python
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

You get:

- Proper scaling
- No data leakage
- Clean code
- Easy reuse
- Safe production workflow

This is **industry standard**.

---

# Big Picture — Topic Philosophy

Training vs Testing is not just about splitting.

It’s about:

- Avoiding data leakage
- Preserving class balance
- Testing stability
- Preventing false confidence

Most Machine Learning failures in the real world are **not because models are bad**.

They happen because **evaluation was sloppy**.

---

# Clean Summary


---

## Multiple Seeds

Tests **model stability across different splits**.

---

## `stratify=y`

Keeps **class balance consistent across train/test**.

---

## Scaling

Normalizes feature ranges so **large numbers do not dominate learning**.

---

## Pipeline

Prevents **data leakage** and enforces **correct training workflow**.